# Evaluation: AUC gate on the stream holdout

In [ ]:
import pathlib
import tarfile
import tempfile
from io import BytesIO

import boto3
import joblib
import pandas as pd
from electricity_price_streaming.feature_contract import (
    FEATURE_COLUMNS,
    LABEL_COLUMN,
)
from feast import FeatureStore
from sklearn.metrics import accuracy_score, roc_auc_score

BUCKET = "electricity-price-streaming"
s3r = boto3.resource(
    "s3", endpoint_url="http://electricity-price-streaming-moto:5000"
)
jobs = sorted(
    o.key
    for o in s3r.Bucket(BUCKET).objects.filter(Prefix="output/")
    if o.key.endswith("model.tar.gz")
)
buf = BytesIO()
s3r.Bucket(BUCKET).download_fileobj(jobs[-1], buf)
buf.seek(0)
tmp = pathlib.Path(tempfile.mkdtemp())
with tarfile.open(fileobj=buf) as t:
    t.extractall(tmp)
model = joblib.load(tmp / "model.joblib")

store = FeatureStore(repo_path="feature_repo")
entities = pd.read_parquet("data/processed/direction_labels.parquet").sample(
    frac=0.2, random_state=7
)
df = store.get_historical_features(
    entity_df=entities,
    features=[f"market_profile:{c}" for c in FEATURE_COLUMNS]
    + [f"direction_labels:{LABEL_COLUMN}"],
).to_df()
probs = model.predict_proba(df[FEATURE_COLUMNS])[:, 1]
print(
    {
        "auc": float(roc_auc_score(df[LABEL_COLUMN], probs)),
        "acc": float(accuracy_score(df[LABEL_COLUMN], probs > 0.5)),
    }
)

In [ ]:
from sklearn.metrics import roc_auc_score

assert roc_auc_score(df[LABEL_COLUMN], probs) >= 0.7, "AUC gate failed"
"gate passed"